<a href="https://colab.research.google.com/github/Nevethawei/my-first-code/blob/main/dl_7th.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, TimeDistributed
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

# 1. Toy Dataset (word + POS tags)

sentences = [
    ["the", "cat", "sat"],
    ["the", "dog", "barked"],
    ["a", "man", "runs"],
    ["the", "woman", "eats"],
]

pos_tags = [
    ["DET", "NOUN", "VERB"],
    ["DET", "NOUN", "VERB"],
    ["DET", "NOUN", "VERB"],
    ["DET", "NOUN", "VERB"],
]

# 2. Vocabulary Creation

words = list(set(w for s in sentences for w in s))
tags = list(set(t for s in pos_tags for t in s))

word2idx = {w: i + 2 for i, w in enumerate(words)}  # reserve 0,1 for PAD/UNK
word2idx["PAD"] = 0
word2idx["UNK"] = 1

tag2idx = {t: i for i, t in enumerate(tags)}

idx2tag = {i: t for t, i in tag2idx.items()}

vocab_size = len(word2idx)
tag_size = len(tag2idx)
max_len = max(len(s) for s in sentences)

# 3. Convert words/tags to integers
X = [[word2idx.get(w, word2idx["UNK"]) for w in s] for s in sentences]
y = [[tag2idx[t] for t in s] for s in pos_tags]

X = pad_sequences(X, maxlen=max_len, padding="post")
y = pad_sequences(y, maxlen=max_len, padding="post")

y = [to_categorical(i, num_classes=tag_size) for i in y]
y = np.array(y)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

# 4. Build Seq2Seq Model
input_layer = Input(shape=(max_len,))
embedding = Embedding(input_dim=vocab_size, output_dim=64, input_length=max_len)(input_layer)
lstm = LSTM(64, return_sequences=True)(embedding)
output = TimeDistributed(Dense(tag_size, activation="softmax"))(lstm)

model = Model(input_layer, output)
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

# 5. Train Model
model.fit(X_train, y_train, batch_size=2, epochs=15, validation_data=(X_test, y_test))

# 6. Prediction Example
test_sentence = ["the", "cat", "runs"]
test_seq = [word2idx.get(w, word2idx["UNK"]) for w in test_sentence]
test_seq = pad_sequences([test_seq], maxlen=max_len, padding="post")

pred = model.predict(test_seq)
pred_tags = [idx2tag[np.argmax(p)] for p in pred[0]]

print("Sentence:", test_sentence)
print("Predicted POS:", pred_tags)





/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 3)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 3, 64)          │           768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 3, 64)          │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed                │ (None, 3, 3)           │           195 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,987 (132.76 KB)

 Trainable params: 33,987 (132.76 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 432ms/step - accuracy: 0.1296 - loss: 1.0986 - val_accuracy: 0.3333 - val_loss: 1.0992
Epoch 2/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.2037 - loss: 1.0952 - val_accuracy: 0.6667 - val_loss: 1.0973
Epoch 3/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.5370 - loss: 1.0905 - val_accuracy: 0.6667 - val_loss: 1.0955
Epoch 4/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - accuracy: 0.5926 - loss: 1.0871 - val_accuracy: 0.6667 - val_loss: 1.0936
Epoch 5/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step - accuracy: 0.5926 - loss: 1.0828 - val_accuracy: 0.6667 - val_loss: 1.0915
Epoch 6/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step - accuracy: 0.5370 - loss: 1.0780 - val_accuracy: 0.6667 - val_loss: 1.0894
Epoch 7/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.5370 - loss: 1.0732 - val_accuracy: 0.6667 - val_loss: 1.0870
Epoch 8/15
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.6667 - loss: 1.0679 - val_accuracy: 0.6667 - val_loss: 1.084

In [ ]:
import numpy as np
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense

# 1. Sample English–French dataset
input_texts = [
    "hi",
    "how are you",
    "i am fine",
    "thank you",
    "good morning"
]

target_texts = [
    "\t salut \n", # \t = start token, \n = end token
    "\t comment ça va \n",
    "\t je vais bien \n",
    "\t merci \n",
    "\t bonjour \n"
]

# 2. Create character sets
input_characters = sorted(list(set("".join(input_texts))))
target_characters = sorted(list(set("".join(target_texts))))

num_encoder_tokens = len(input_characters)
num_decoder_tokens = len(target_characters)
max_encoder_seq_length = max([len(txt) for txt in input_texts])
max_decoder_seq_length = max([len(txt) for txt in target_texts])

print("Input tokens:", num_encoder_tokens)
print("Output tokens:", num_decoder_tokens)

# 3. Create dictionaries
input_token_index = dict([(char, i) for i, char in enumerate(input_characters)])
target_token_index = dict([(char, i) for i, char in enumerate(target_characters)])

# 4. Vectorize the input and output data
encoder_input_data = np.zeros(
    (len(input_texts), max_encoder_seq_length, num_encoder_tokens), dtype="float32")
decoder_input_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32")
decoder_target_data = np.zeros(
    (len(input_texts), max_decoder_seq_length, num_decoder_tokens), dtype="float32")

for i, (input_text, target_text) in enumerate(zip(input_texts, target_texts)):
    for t, char in enumerate(input_text):
        encoder_input_data[i, t, input_token_index[char]] = 1.0
    for t, char in enumerate(target_text):
        decoder_input_data[i, t, target_token_index[char]] = 1.0
        if t > 0:
            decoder_target_data[i, t - 1, target_token_index[char]] = 1.0

# 5. Build Encoder–Decoder Model
latent_dim = 256

# Encoder
encoder_inputs = Input(shape=(None, num_encoder_tokens))
encoder_lstm = LSTM(latent_dim, return_state=True)
_, state_h, state_c = encoder_lstm(encoder_inputs)
encoder_states = [state_h, state_c]

# Decoder
decoder_inputs = Input(shape=(None, num_decoder_tokens))
decoder_lstm = LSTM(latent_dim, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_inputs, initial_state=encoder_states)
decoder_dense = Dense(num_decoder_tokens, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# Combined Model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='rmsprop', loss='categorical_crossentropy')

# 6. Train Model
model.fit([encoder_input_data, decoder_input_data],
          decoder_target_data,
          batch_size=5,
          epochs=300,
          validation_split=0.2)

# 7. Inference Models for Testing
encoder_model = Model(encoder_inputs, encoder_states)

decoder_state_input_h = Input(shape=(latent_dim,))
decoder_state_input_c = Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]
decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_inputs, initial_state=decoder_states_inputs)
decoder_states = [state_h, state_c]
decoder_outputs = decoder_dense(decoder_outputs)
decoder_model = Model(
    [decoder_inputs] + decoder_states_inputs,
    [decoder_outputs] + decoder_states)

# 8. Reverse dictionaries for decoding
reverse_input_char_index = dict((i, char) for char, i in input_token_index.items())
reverse_target_char_index = dict((i, char) for char, i in target_token_index.items())

# 9. Decode function
def decode_sequence(input_seq):
    states_value = encoder_model.predict(input_seq)
    target_seq = np.zeros((1, 1, num_decoder_tokens))
    target_seq[0, 0, target_token_index["\t"]] = 1.0

    decoded_sentence = ""
    while True:
        output_tokens, h, c = decoder_model.predict([target_seq] + states_value)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_char = reverse_target_char_index[sampled_token_index]
        decoded_sentence += sampled_char

        if (sampled_char == "\n" or len(decoded_sentence) > max_decoder_seq_length):
            break
        target_seq = np.zeros((1, 1, num_decoder_tokens))
        target_seq[0, 0, sampled_token_index] = 1.0
        states_value = [h, c]
    return decoded_sentence

# 10. Test translation
for seq_index in range(len(input_texts)):
    input_seq = encoder_input_data[seq_index: seq_index + 1]
    decoded_sentence = decode_sequence(input_seq)
    print("Input:", input_texts[seq_index])
    print("Predicted translation:", decoded_sentence)


Input tokens: 17
Output tokens: 19
Epoch 1/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - loss: 2.0318 - val_loss: 1.7266
Epoch 2/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step - loss: 2.0103 - val_loss: 1.7223
Epoch 3/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - loss: 1.9921 - val_loss: 1.7180
Epoch 4/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step - loss: 1.9724 - val_loss: 1.7128
Epoch 5/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step - loss: 1.9474 - val_loss: 1.7054
Epoch 6/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 149ms/step - loss: 1.9095 - val_loss: 1.6936
Epoch 7/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step - loss: 1.8448 - val_loss: 1.6889
Epoch 8/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 150ms/step - loss: 1.7884 - val_loss: 1.7017
Epoch 9/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 143ms/step - loss: 1.8156 - val_loss: 1.7621
Epoch 10/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 145ms/step - loss: 1.7508 - val_loss: 1.7455
Epoch 11/300
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step - loss: 1.7954 - val_loss: 1.8097
Epoch 12/300
1/1 ━━━━━━━━━━━━━━